# 02 — Crater Floor QC and Recovery

This notebook preserves the two recovery passes used to expand the Crater Floor
dataset from the initially usable observations to the final 185-observation
Crater Floor dataset.

Historical lineage:

- 209 Crater Floor WAV observations were processed.
- 82 were set aside for additional onset/QC review.
- 43 were recovered in the first pass, producing the retained 170-row intermediate dataset.
- 15 more were recovered in the second pass, producing `LIBS_acoustic_meta_sheet_v3_185.csv`.

Important limitation: the original 127-row geological metadata sheet used in the
historical first-pass merge was not retained. Therefore this cleaned notebook can
recompute the first-pass recovered metrics, but it treats the retained
`LIBS_acoustic_meta_sheet_expanded.csv` (170 rows) as the authoritative intermediate
for continuing into pass 2.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import find_peaks, windows
from scipy.stats import linregress

## Paths

In [ ]:
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")

WAV_DIR = PROJECT_ROOT / "data" / "raw" / "crater_floor_wav"
QC_DIR = PROJECT_ROOT / "data" / "qc"
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

QC_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RECOVER_DECISIONS_PATH = QC_DIR / "Acoustic Wave Data - Recover.csv"
ONSET_RECOVERY_SUMMARY_PATH = QC_DIR / "onset_recovery_summary.csv"
PASS2_SUMMARY_PATH = QC_DIR / "recovery_pass_2_summary.csv"

FIRST_PASS_METRICS_PATH = INTERMEDIATE_DIR / "recovered_metrics_first_pass.csv"
SECOND_PASS_METRICS_PATH = INTERMEDIATE_DIR / "recovered_metrics_pass2.csv"

# Retained historical intermediate after the first recovery pass.
EXPANDED_170_PATH = INTERMEDIATE_DIR / "LIBS_acoustic_meta_sheet_expanded.csv"

FINAL_185_PATH = PROCESSED_DIR / "LIBS_acoustic_meta_sheet_v3_185.csv"

print("Crater Floor WAVs:", WAV_DIR.resolve())
print("First-pass decisions:", RECOVER_DECISIONS_PATH.resolve())
print("170-row intermediate:", EXPANDED_170_PATH.resolve())
print("Final 185-row output:", FINAL_185_PATH.resolve())

## QC records

These CSVs preserve the manual review process. They are not regenerated here because
they reflect human inspection of candidate onset detections.

In [ ]:
for label, path in {
    "First-pass onset sweep": ONSET_RECOVERY_SUMMARY_PATH,
    "Second-pass onset sweep": PASS2_SUMMARY_PATH,
}.items():
    if path.exists():
        temp = pd.read_csv(path)
        print(f"{label}: {temp.shape}")
    else:
        print(f"{label}: not found at {path}")

## First recovery pass

The first pass tested four onset-detection parameter sets. The manually selected
method for each recoverable file is stored in `Acoustic Wave Data - Recover.csv`.
The code below recomputes acoustic metrics using those retained decisions.

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

expected_shot_time = 0.073
next_shot_start = 0.133
response_window = 0.010

fit_db_top = -7
fit_db_bottom = -15
t_c = 0.002

noise_window = 0.003
min_peak_distance_s = 0.010
min_run_s = 0.00005

usable_band = (1000, 50000)
low_band = (1000, 10000)
high_band = (10000, 30000)

parameter_sets = {
    "Original": {
        "search_half_width": 0.005,
        "threshold_sigma": 4,
        "height_sigma": 4,
        "prominence_sigma": 3,
        "backtrack_window_s": 0.003,
        "backtrack_sigma": 4,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 0.4
    },
    "wider 3p5 sigma": {
        "search_half_width": 0.010,
        "threshold_sigma": 3.5,
        "height_sigma": 3.5,
        "prominence_sigma": 2.5,
        "backtrack_window_s": 0.005,
        "backtrack_sigma": 3.5,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 0.8
    },
    "wider 3 sigma": {
        "search_half_width": 0.010,
        "threshold_sigma": 3,
        "height_sigma": 3,
        "prominence_sigma": 2,
        "backtrack_window_s": 0.006,
        "backtrack_sigma": 3,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 1.0
    },
    "very wide soft": {
        "search_half_width": 0.015,
        "threshold_sigma": 3,
        "height_sigma": 3,
        "prominence_sigma": 1.5,
        "backtrack_window_s": 0.008,
        "backtrack_sigma": 3,
        "smooth_window_s": 0.00008,
        "flag_lag_ms": 1.2
    }
}

# ============================================================
# FUNCTIONS
# ============================================================

def detect_onset(x, fs, settings):
    x = x.astype(np.float64)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    abs_x = np.abs(x)

    smooth_n = max(1, int(round(settings["smooth_window_s"] * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")

    search_start = expected_shot_time - settings["search_half_width"]
    search_end = expected_shot_time + settings["search_half_width"]

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    inoise1 = isearch0
    inoise0 = max(0, int(round((search_start - noise_window) * fs)))

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    min_height = noise_mean + settings["height_sigma"] * noise_std
    min_prominence = settings["prominence_sigma"] * noise_std
    min_peak_distance = int(round(min_peak_distance_s * fs))

    search_env = env[isearch0:isearch1]

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance
    )

    if len(peaks) == 0:
        return None, x

    peak_index = isearch0 + peaks[0]
    peak_time = peak_index / fs

    backtrack_samples = int(round(settings["backtrack_window_s"] * fs))
    back_start = max(isearch0, peak_index - backtrack_samples)

    noise_back_start = max(0, back_start - int(round(0.005 * fs)))
    noise_back_end = back_start
    local_noise = env[noise_back_start:noise_back_end]

    if len(local_noise) > 0:
        onset_threshold = np.mean(local_noise) + settings["backtrack_sigma"] * np.std(local_noise)
    else:
        onset_threshold = noise_mean + settings["threshold_sigma"] * noise_std

    search_back_env = env[back_start:peak_index]
    above = search_back_env > onset_threshold

    min_run_samples = max(1, int(round(min_run_s * fs)))
    onset_index = None

    for i in range(len(above) - min_run_samples + 1):
        if np.all(above[i:i + min_run_samples]):
            onset_index = back_start + i
            break

    if onset_index is None:
        onset_index = peak_index

    onset_time = onset_index / fs
    response_stop = onset_time + response_window

    if response_stop > next_shot_start:
        response_stop = next_shot_start

    lag_ms = (peak_index - onset_index) / fs * 1000

    return {
        "onset_index": onset_index,
        "onset_time": onset_time,
        "peak_time": peak_time,
        "lag_ms": lag_ms,
        "response_stop": response_stop
    }, x


def compute_time_metrics(segment, fs):
    t = np.arange(len(segment)) / fs
    energy = segment ** 2

    edc = np.cumsum(energy[::-1])[::-1]
    edc_norm = edc / np.max(edc)
    edc_db = 10 * np.log10(edc_norm + 1e-20)

    fit_mask = (edc_db <= fit_db_top) & (edc_db >= fit_db_bottom)

    if np.sum(fit_mask) < 2:
        slope = np.nan
        r2 = np.nan
        drop_time = np.nan
    else:
        slope, intercept, r_value, p_value, std_err = linregress(
            t[fit_mask],
            edc_db[fit_mask]
        )
        r2 = r_value ** 2
        db_drop = abs(fit_db_bottom - fit_db_top)
        drop_time = db_drop / abs(slope) if slope != 0 else np.nan

    i_c = int(round(t_c * fs))

    early_energy = np.sum(energy[:i_c])
    late_energy = np.sum(energy[i_c:])

    C2 = 10 * np.log10(early_energy / late_energy) if late_energy > 0 else np.nan

    return slope, r2, drop_time, C2


def compute_fft_metrics(segment, fs):
    segment = segment - np.mean(segment)
    hann = windows.hann(len(segment))
    seg_w = segment * hann

    freqs = np.fft.rfftfreq(len(seg_w), d=1/fs)
    X = np.fft.rfft(seg_w)
    power = np.abs(X) ** 2

    usable_mask = (freqs >= usable_band[0]) & (freqs <= usable_band[1])
    f = freqs[usable_mask]
    p = power[usable_mask]

    if len(p) == 0 or np.sum(p) == 0:
        return {
            "Spectral Centroid (Hz)": np.nan,
            "Spectral Bandwidth (Hz)": np.nan,
            "Peak Frequency (Hz)": np.nan,
            "Rolloff 85% (Hz)": np.nan,
            "Low Power 1-10 kHz": np.nan,
            "High Power 10-30 kHz": np.nan,
            "High/Low Ratio": np.nan,
            "High Frequency Fraction": np.nan,
            "Total FFT Power": np.nan
        }

    p_sum = np.sum(p)

    centroid = np.sum(f * p) / p_sum
    bandwidth = np.sqrt(np.sum(((f - centroid) ** 2) * p) / p_sum)
    peak_freq = f[np.argmax(p)]

    cumulative = np.cumsum(p)
    rolloff_85 = f[np.where(cumulative >= 0.85 * p_sum)[0][0]]

    low_mask = (freqs >= low_band[0]) & (freqs < low_band[1])
    high_mask = (freqs >= high_band[0]) & (freqs < high_band[1])

    low_power = np.sum(power[low_mask])
    high_power = np.sum(power[high_mask])
    total_power = np.sum(power)

    return {
        "Spectral Centroid (Hz)": centroid,
        "Spectral Bandwidth (Hz)": bandwidth,
        "Peak Frequency (Hz)": peak_freq,
        "Rolloff 85% (Hz)": rolloff_85,
        "Low Power 1-10 kHz": low_power,
        "High Power 10-30 kHz": high_power,
        "High/Low Ratio": high_power / low_power if low_power > 0 else np.nan,
        "High Frequency Fraction": high_power / total_power if total_power > 0 else np.nan,
        "Total FFT Power": total_power
    }

In [ ]:
# ============================================================

recover_df = pd.read_csv(RECOVER_DECISIONS_PATH)

recover_df = recover_df.dropna(subset=["Best Onset"]).copy()
recover_df = recover_df[recover_df["Best Onset"].astype(str).str.upper() != "N/A"]

print("Files to recover:", len(recover_df))
print(recover_df["Best Onset"].value_counts())

# ============================================================
# PROCESS RECOVERED FILES
# ============================================================

rows = []

for _, row in recover_df.iterrows():
    filename = row["File Name"]
    method = row["Best Onset"]

    wav_path = WAV_DIR / filename

    if not wav_path.exists():
        print("Missing:", filename)
        continue

    settings = parameter_sets[method]

    fs, x_raw = wavfile.read(wav_path)
    detection, x = detect_onset(x_raw, fs, settings)

    if detection is None:
        print("Failed:", filename)
        continue

    i0 = detection["onset_index"]
    i1 = int(round(detection["response_stop"] * fs))
    segment = x[i0:i1]

    if len(segment) == 0:
        print("Empty segment:", filename)
        continue

    slope, r2, drop_time, C2 = compute_time_metrics(segment, fs)
    fft_metrics = compute_fft_metrics(segment, fs)

    rows.append({
        "File Name": filename,
        "Onset (s)": detection["onset_time"],
        "Slope (dB/s)": slope,
        "R^2": r2,
        "Drop Time (s)": drop_time,
        "C2 (dB)": C2,
        "Recovery Method": method,
        **fft_metrics
    })

recovered_metrics = pd.DataFrame(rows)
recovered_metrics.to_csv(FIRST_PASS_METRICS_PATH, index=False)

print("Recovered metrics saved to:")
print(FIRST_PASS_METRICS_PATH)
print("Recovered rows:", len(recovered_metrics))

The historical workflow then merged these recovered observations into the original
127-row Crater Floor metadata sheet to create
`LIBS_acoustic_meta_sheet_expanded.csv` (170 rows).

Because that original 127-row metadata sheet is no longer retained, the existing
170-row CSV is preserved as the authoritative intermediate rather than reconstructing
that merge from incomplete inputs.

## Second recovery pass

The remaining difficult files were manually reviewed across candidate expected onset
times. The 15 accepted file/onset-time pairs below are retained directly from the
final recovery code.

In [ ]:
# ============================================================
# PASS 2 ACCEPTED FILES

# ============================================================

pass2_recoveries = [
    ("scam_0071_0673238158_718_ca0_scam05071_hadahastsaa__________01p01.wav", 0.070),
    ("scam_0104_0676170543_585_ca0_scam05104_ad_ees_eez___________01p01.wav", 0.085),
    ("scam_0113_0676966786_559_ca0_scam01113_mussih_______________01p01.wav", 0.085),
    ("scam_0113_0676966938_600_ca0_scam01113_mussih_______________02p01.wav", 0.085),
    ("scam_0183_0683183544_821_ca0_scam01183_sauzeries_hautes_____01p01.wav", 0.085),
    ("scam_0183_0683183736_826_ca0_scam01183_sauzeries_hautes_____02p01.wav", 0.085),
    ("scam_0213_0685844643_151_ca0_scam04213_moustiers_sainte_mar_01p01.wav", 0.076),
    ("scam_0250_0689128907_101_ca0_scam01250_hotel________________02p01.wav", 0.085),
    ("scam_0274_0691263225_224_ca0_scam03274_chasteuil____________01p01.wav", 0.085),
    ("scam_0286_0692324948_436_ca0_scam01286_bezaudun_____________01p01.wav", 0.085),
    ("scam_0312_0694632787_748_ca0_scam01312_riolan_312___________02p01.wav", 0.080),
    ("scam_0335_0696680688_405_ca0_scam01335_chabran______________01p01.wav", 0.085),
    ("scam_0335_0696680738_320_ca0_scam01335_chabran______________02p01.wav", 0.085),
    ("scam_0343_0697389227_661_ca0_scam02343_chanolles____________01p01.wav", 0.085),
    ("scam_0361_0698983364_602_ca0_scam01361_naanazwod____________02p01.wav", 0.076),
]

# ============================================================
# SETTINGS
# ============================================================

next_shot_start = 0.133
response_window = 0.010

fit_db_top = -7
fit_db_bottom = -15
t_c = 0.002

noise_window = 0.003
min_peak_distance_s = 0.010
min_run_s = 0.00005

usable_band = (1000, 50000)
low_band = (1000, 10000)
high_band = (10000, 30000)

# Pass 2 detector settings
settings = {
    "search_half_width": 0.008,
    "threshold_sigma": 3.5,
    "height_sigma": 3.5,
    "prominence_sigma": 2.5,
    "backtrack_window_s": 0.006,
    "backtrack_sigma": 3.5,
    "smooth_window_s": 0.00005,
    "flag_lag_ms": 1.0
}

# ============================================================
# FUNCTIONS
# ============================================================

def prepare_audio(x_raw):
    x = x_raw.astype(np.float64)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    return x


def make_envelope(x, fs, smooth_window_s):
    abs_x = np.abs(x)
    smooth_n = max(1, int(round(smooth_window_s * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")
    return env


def detect_onset(x, env, fs, settings, expected_shot_time):
    search_start = expected_shot_time - settings["search_half_width"]
    search_end = expected_shot_time + settings["search_half_width"]

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    inoise1 = isearch0
    inoise0 = max(0, int(round((search_start - noise_window) * fs)))

    if isearch1 <= isearch0 or inoise1 <= inoise0:
        return None

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    peak_threshold = noise_mean + settings["threshold_sigma"] * noise_std
    min_height = noise_mean + settings["height_sigma"] * noise_std
    min_prominence = settings["prominence_sigma"] * noise_std
    min_peak_distance = int(round(min_peak_distance_s * fs))

    search_env = env[isearch0:isearch1]

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance
    )

    if len(peaks) == 0:
        return None

    peak_index = isearch0 + peaks[0]
    peak_time = peak_index / fs

    backtrack_samples = int(round(settings["backtrack_window_s"] * fs))
    back_start = max(isearch0, peak_index - backtrack_samples)

    noise_back_start = max(0, back_start - int(round(0.005 * fs)))
    noise_back_end = back_start
    local_noise = env[noise_back_start:noise_back_end]

    if len(local_noise) > 0:
        onset_threshold = np.mean(local_noise) + settings["backtrack_sigma"] * np.std(local_noise)
    else:
        onset_threshold = peak_threshold

    search_back_env = env[back_start:peak_index]
    above = search_back_env > onset_threshold

    min_run_samples = max(1, int(round(min_run_s * fs)))
    onset_index = None

    for i in range(len(above) - min_run_samples + 1):
        if np.all(above[i:i + min_run_samples]):
            onset_index = back_start + i
            break

    forced_peak = False

    if onset_index is None:
        onset_index = peak_index
        forced_peak = True

    onset_time = onset_index / fs
    response_stop = onset_time + response_window

    if response_stop > next_shot_start:
        response_stop = next_shot_start

    lag_ms = (peak_index - onset_index) / fs * 1000

    flagged = forced_peak or lag_ms > settings["flag_lag_ms"]

    return {
        "onset_index": onset_index,
        "onset_time": onset_time,
        "peak_time": peak_time,
        "peak_onset_lag_ms": lag_ms,
        "response_stop": response_stop,
        "flagged": flagged,
        "forced_peak": forced_peak
    }


def compute_time_metrics(segment, fs):
    t = np.arange(len(segment)) / fs
    energy = segment ** 2

    edc = np.cumsum(energy[::-1])[::-1]

    if np.max(edc) == 0:
        return np.nan, np.nan, np.nan, np.nan

    edc_norm = edc / np.max(edc)
    edc_db = 10 * np.log10(edc_norm + 1e-20)

    fit_mask = (edc_db <= fit_db_top) & (edc_db >= fit_db_bottom)

    if np.sum(fit_mask) < 2:
        slope = np.nan
        r2 = np.nan
        drop_time = np.nan
    else:
        slope, intercept, r_value, p_value, std_err = linregress(
            t[fit_mask],
            edc_db[fit_mask]
        )
        r2 = r_value ** 2
        db_drop = abs(fit_db_bottom - fit_db_top)
        drop_time = db_drop / abs(slope) if slope != 0 else np.nan

    i_c = int(round(t_c * fs))

    if i_c >= len(segment):
        return slope, r2, drop_time, np.nan

    early_energy = np.sum(energy[:i_c])
    late_energy = np.sum(energy[i_c:])

    C2 = 10 * np.log10(early_energy / late_energy) if late_energy > 0 else np.nan

    return slope, r2, drop_time, C2


def compute_fft_metrics(segment, fs):
    segment = segment - np.mean(segment)

    hann = windows.hann(len(segment))
    seg_w = segment * hann

    freqs = np.fft.rfftfreq(len(seg_w), d=1/fs)
    X = np.fft.rfft(seg_w)
    power = np.abs(X) ** 2

    usable_mask = (freqs >= usable_band[0]) & (freqs <= usable_band[1])
    f = freqs[usable_mask]
    p = power[usable_mask]

    if len(p) == 0 or np.sum(p) == 0:
        return {
            "Spectral Centroid (Hz)": np.nan,
            "Spectral Bandwidth (Hz)": np.nan,
            "Peak Frequency (Hz)": np.nan,
            "Rolloff 85% (Hz)": np.nan,
            "Low Power 1-10 kHz": np.nan,
            "High Power 10-30 kHz": np.nan,
            "High/Low Ratio": np.nan,
            "High Frequency Fraction": np.nan,
            "Total FFT Power": np.nan
        }

    p_sum = np.sum(p)

    centroid = np.sum(f * p) / p_sum
    bandwidth = np.sqrt(np.sum(((f - centroid) ** 2) * p) / p_sum)
    peak_freq = f[np.argmax(p)]

    cumulative = np.cumsum(p)
    rolloff_85 = f[np.where(cumulative >= 0.85 * p_sum)[0][0]]

    low_mask = (freqs >= low_band[0]) & (freqs < low_band[1])
    high_mask = (freqs >= high_band[0]) & (freqs < high_band[1])

    low_power = np.sum(power[low_mask])
    high_power = np.sum(power[high_mask])
    total_power = np.sum(power)

    return {
        "Spectral Centroid (Hz)": centroid,
        "Spectral Bandwidth (Hz)": bandwidth,
        "Peak Frequency (Hz)": peak_freq,
        "Rolloff 85% (Hz)": rolloff_85,
        "Low Power 1-10 kHz": low_power,
        "High Power 10-30 kHz": high_power,
        "High/Low Ratio": high_power / low_power if low_power > 0 else np.nan,
        "High Frequency Fraction": high_power / total_power if total_power > 0 else np.nan,
        "Total FFT Power": total_power
    }

# ============================================================
# PROCESS PASS 2 FILES
# ============================================================

rows = []

for filename, expected_time in pass2_recoveries:
    wav_path = WAV_DIR / filename

    if not wav_path.exists():
        print("Missing:", filename)
        continue

    fs, x_raw = wavfile.read(wav_path)

    x = prepare_audio(x_raw)
    env = make_envelope(x, fs, settings["smooth_window_s"])

    detection = detect_onset(
        x=x,
        env=env,
        fs=fs,
        settings=settings,
        expected_shot_time=expected_time
    )

    if detection is None:
        print("Failed detection:", filename)
        continue

    i0 = detection["onset_index"]
    i1 = int(round(detection["response_stop"] * fs))

    segment = x[i0:i1]

    if len(segment) == 0:
        print("Empty segment:", filename)
        continue

    slope, r2, drop_time, C2 = compute_time_metrics(segment, fs)
    fft_metrics = compute_fft_metrics(segment, fs)

    rows.append({
        "File Name": filename,
        "Flagged": detection["flagged"],
        "Sample Rate (Hz)": fs,
        "Onset (s)": detection["onset_time"],
        "Peak Time (s)": detection["peak_time"],
        "Peak-Onset Lag (ms)": detection["peak_onset_lag_ms"],
        "Slope (dB/s)": slope,
        "R^2": r2,
        "Drop Time (s)": drop_time,
        "C2 (dB)": C2,
        "Recovery Method": "pass2_expected_time_sweep",
        "Pass2 Expected Time (s)": expected_time,
        **fft_metrics
    })

pass2_df = pd.DataFrame(rows)
pass2_df.to_csv(SECOND_PASS_METRICS_PATH, index=False)

print("Pass 2 recovered metrics saved to:")
print(SECOND_PASS_METRICS_PATH)
print("Pass 2 rows:", len(pass2_df))

# ============================================================
# APPEND TO EXISTING META SHEET
# ============================================================

meta_df = pd.read_csv(EXPANDED_170_PATH)

combined_df = pd.concat(
    [meta_df, pass2_df],
    ignore_index=True,
    sort=False
)

combined_df = combined_df.drop_duplicates(
    subset=["File Name"],
    keep="first"
)

combined_df.to_csv(FINAL_185_PATH, index=False)

print()
print("Final meta sheet saved to:")
print(FINAL_185_PATH)
print("Old rows:", len(meta_df))
print("Pass 2 rows added:", len(pass2_df))
print("Final rows:", len(combined_df))

## Expected result

With the original Crater Floor WAV collection and the retained 170-row intermediate,
this notebook should append 15 pass-2 recoveries and create the final
`LIBS_acoustic_meta_sheet_v3_185.csv`.